# MARTA Weekly-Cadence PPO Grid — seeds 1-3 (crypto5 panel)

Fills in the multi-seed variance estimate for `rl/train_weekly_crypto5_grid.py`
(BTC/ETH/BNB/LTC/XRP x 4 conditions x seeds 1-3). Seed-0 pilot already ran locally.

**Settings before running:** Accelerator = None (CPU), Internet = On, attach the
`marta-weekly-ppo-pkg` dataset via Add Data.

**Change `ASSET` in the training cell below before each "Save & Run All (Commit)"** —
one commit per asset (~2.5h each), per `kaggle_weekly_ppo_grid_instructions.md`.

In [ ]:
import os, glob

candidates = sorted(d for d in glob.glob("/kaggle/input/*") if os.path.isdir(d))
print("Top-level /kaggle/input/ entries:", candidates)

def _find_pkg_root(base, depth=0, max_depth=6):
    if os.path.exists(os.path.join(base, "rl", "train.py")):
        return base
    if depth >= max_depth:
        return None
    for sub in sorted(glob.glob(os.path.join(base, "*"))):
        if os.path.isdir(sub):
            found = _find_pkg_root(sub, depth + 1, max_depth)
            if found:
                return found
    return None

src = None
for d in candidates:
    src = _find_pkg_root(d)
    if src:
        break

assert src is not None, (
    f"Could not find rl/train.py under any /kaggle/input/* entry (searched up to "
    f"6 levels deep) — attach the marta-weekly-ppo-pkg dataset via Add Data. "
    f"Found: {candidates}"
)
print("Using package at:", src)

!rm -rf /kaggle/working/marta
!cp -r {src} /kaggle/working/marta
%cd /kaggle/working/marta
!mkdir -p rl/checkpoints data/results/weekly_grid_crypto5

In [ ]:
!pip install -q stable-baselines3==2.8.0 gymnasium==1.2.3 mlflow-skinny==3.12.0 rich==15.0.0 arch==8.0.0

import os
# mlflow 3.x deprecated the local filesystem tracking store by default (same fix
# dashboard/docker-compose.yml already needed) — this notebook has no tracking server.
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

# arch==8.0.0 matches the local conda env that produced the seed-0 pilot checkpoints —
# without it, rl/env.py:169 silently falls back to a rolling-std vol estimate instead of
# a real GARCH(1,1) fit, which would make seeds 1-3 methodologically inconsistent with
# seed-0 within the same (asset, condition) cell.

In [ ]:
# Sanity check — confirms the package's data loads correctly and the path-anchoring
# fix works from /kaggle/working/marta (a different cwd than this ever ran from
# locally), before spending 12+ minutes finding that out via a failed training cell.
import sys
sys.path.insert(0, "/kaggle/working/marta")

import rl.train as t
print("MARTA_ROOT resolved to:", t.MARTA_ROOT)  # must be /kaggle/working/marta
assert str(t.MARTA_ROOT) == "/kaggle/working/marta"

from rl.train_weekly_crypto5_grid import build_weekly_config
from rl.train import load_data
cfg = build_weekly_config("BTC_USDT", "raw", seed=1)
aligned_dfs, regime_stream, encoder = load_data(cfg)
print("Data OK —", len(aligned_dfs["BTC_USDT"]), "BTC_USDT rows, encoder:", encoder)

In [ ]:
ASSET = "BTC_USDT"  # <- change this each of the 5 times you run this notebook

!cd /kaggle/working/marta && python rl/train_weekly_crypto5_grid.py --asset {ASSET} --label A --seed 1
!cd /kaggle/working/marta && python rl/train_weekly_crypto5_grid.py --asset {ASSET} --label A --seed 2
!cd /kaggle/working/marta && python rl/train_weekly_crypto5_grid.py --asset {ASSET} --label A --seed 3
!cd /kaggle/working/marta && python rl/train_weekly_crypto5_grid.py --asset {ASSET} --label B --seed 1
!cd /kaggle/working/marta && python rl/train_weekly_crypto5_grid.py --asset {ASSET} --label B --seed 2
!cd /kaggle/working/marta && python rl/train_weekly_crypto5_grid.py --asset {ASSET} --label B --seed 3
!cd /kaggle/working/marta && python rl/train_weekly_crypto5_grid.py --asset {ASSET} --label C --seed 1
!cd /kaggle/working/marta && python rl/train_weekly_crypto5_grid.py --asset {ASSET} --label C --seed 2
!cd /kaggle/working/marta && python rl/train_weekly_crypto5_grid.py --asset {ASSET} --label C --seed 3
!cd /kaggle/working/marta && python rl/train_weekly_crypto5_grid.py --asset {ASSET} --label D --seed 1
!cd /kaggle/working/marta && python rl/train_weekly_crypto5_grid.py --asset {ASSET} --label D --seed 2
!cd /kaggle/working/marta && python rl/train_weekly_crypto5_grid.py --asset {ASSET} --label D --seed 3

In [ ]:
!cd /kaggle/working/marta && python rl/train_weekly_crypto5_grid.py --summarize-only